In [3]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [4]:
spark = (SparkSession.builder.appName("Bus Arrival Prediction").config("spark.sql.shuffle.partitions", "4").getOrCreate())

In [5]:
import os
gtfs = {}
for file in os.listdir("Data/Raw"):
    if file.endswith(".txt"):
        name = file.replace(".txt", "")
        gtfs[name] = spark.read.csv(
            f"Data/Raw/{file}",
            header=True,
            inferSchema=True)
print(gtfs.keys())
vehicle = pd.read_csv("Data/Raw/vehicle.csv")
disruption = pd.read_csv("Data/Raw/distrubtion.csv")

dict_keys(['agency', 'calendar', 'calendar_dates', 'feed_info', 'routes', 'shapes', 'stops', 'stop_times', 'trips'])


# Cleaning Dataset

In [6]:
vehicle["RecordedTime"] = pd.to_datetime(vehicle["RecordedTime"])
vehicle["Latitude"] = pd.to_numeric(vehicle["Latitude"], errors="coerce")
vehicle["Longitude"] = pd.to_numeric(vehicle["Longitude"], errors="coerce")
vehicle = vehicle.drop_duplicates()
vehicle = vehicle.dropna(subset=["LineRef", "Latitude", "Longitude"])
vehicle.head()

,RecordedTime,LineRef,Operator,VehicleRef,Latitude,Longitude
0,2026-07-02 15:10:13+00:00,96A,A2BR,3303,51.981412,-0.233722
1,2026-07-02 17:33:39+00:00,16A,A2BR,3311,52.073968,0.008972
2,2026-07-02 18:12:15+00:00,811,A2BV,A2BV-24,53.368135,-3.069226
3,2026-07-02 19:16:10+00:00,1A,A2BV,A2BV-RE24_TDZ,53.365285,-3.065045
4,2026-07-02 17:11:17+00:00,106,A2BV,A2BV-YJ10_MHK,53.419548,-3.046595


In [7]:
# Convert datetime columns
disruption["CreationTime"] = pd.to_datetime(
    disruption["CreationTime"],
    format="mixed",
    utc=True
)

disruption["VersionedAtTime"] = pd.to_datetime(
    disruption["VersionedAtTime"],
    format="mixed",
    utc=True
)

# Remove duplicates
disruption = disruption.drop_duplicates()

# Fill missing values
disruption["MiscellaneousReason"] = disruption["MiscellaneousReason"].fillna("Unknown")
disruption["EquipmentReason"] = disruption["EquipmentReason"].fillna("Unknown")
disruption["InfoLinks"] = disruption["InfoLinks"].fillna("None")

print("Disruption Shape:", disruption.shape)

Disruption Shape: (374, 16)


In [8]:
print("\nVehicle Info")
vehicle.info()

print("\nDisruption Info")
disruption.info()

print("\nVehicle Sample")
print(vehicle.head())

print("\nDisruption Sample")
print(disruption.head())


Vehicle Info
<class 'pandas.core.frame.DataFrame'>
Index: 27488 entries, 0 to 27501
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   RecordedTime  27488 non-null  datetime64[ns, UTC]
 1   LineRef       27488 non-null  object             
 2   Operator      27488 non-null  object             
 3   VehicleRef    27488 non-null  object             
 4   Latitude      27488 non-null  float64            
 5   Longitude     27488 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(2), object(3)
memory usage: 1.5+ MB

Disruption Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   CreationTime         374 non-null    datetime64[ns, UTC]
 1   ParticipantRef       374 non-null    object           

In [9]:
# Remove newline characters and extra spaces
disruption = disruption.replace(r'^\s*$', pd.NA, regex=True)

# Fill missing values
disruption["Source"] = disruption["Source"].fillna("Unknown")
disruption["ValidityPeriod"] = disruption["ValidityPeriod"].fillna("Unknown")
disruption["PublicationWindow"] = disruption["PublicationWindow"].fillna("Unknown")
disruption["Consequences"] = disruption["Consequences"].fillna("Unknown")
disruption["InfoLinks"] = disruption["InfoLinks"].fillna("None")

In [10]:
disruption.isnull().sum()

CreationTime           0
ParticipantRef         0
SituationNumber        0
Version                0
Source                 0
VersionedAtTime        0
Progress               0
ValidityPeriod         0
PublicationWindow      0
MiscellaneousReason    0
Planned                0
Summary                0
Description            0
Consequences           0
InfoLinks              0
EquipmentReason        0
dtype: int64

# Feature Selection

In [11]:
gtfs["agency"] = gtfs["agency"].select("agency_id","agency_name","agency_noc")
gtfs["routes"] = gtfs["routes"].select("route_id","agency_id","route_short_name","route_long_name","route_type")
gtfs["trips"] = gtfs["trips"].select("trip_id","route_id","service_id","trip_headsign","direction_id","shape_id","wheelchair_accessible")
gtfs["stop_times"] = gtfs["stop_times"].select(
    "trip_id",
    "arrival_time",
    "departure_time",
    "stop_id",
    "stop_sequence",
    "pickup_type",
    "drop_off_type",
    "timepoint"
)
gtfs["stops"] = gtfs["stops"].select("stop_id","stop_name","stop_lat","stop_lon","wheelchair_boarding")
gtfs["calendar"] = gtfs["calendar"].select("service_id","monday","tuesday","wednesday","thursday","friday","saturday","sunday","start_date","end_date")
vehicle = vehicle[[
    "RecordedTime",
    "LineRef",
    "Operator",
    "VehicleRef",
    "Latitude",
    "Longitude"
]]

In [12]:
disruption = disruption[[
    "CreationTime",
    "ParticipantRef",
    "Progress",
    "MiscellaneousReason",
    "Planned",
    "Summary",
    "Description",
    "EquipmentReason"
]]

In [13]:
for name, df in gtfs.items():
    print(f"\n{name.upper()}")
    print(df.columns)


AGENCY
['agency_id', 'agency_name', 'agency_noc']

CALENDAR
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

CALENDAR_DATES
['service_id', 'date', 'exception_type']

FEED_INFO
['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version']

ROUTES
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

SHAPES
['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled']

STOPS
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'wheelchair_boarding']

STOP_TIMES
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'timepoint']

TRIPS
['trip_id', 'route_id', 'service_id', 'trip_headsign', 'direction_id', 'shape_id', 'wheelchair_accessible']


In [14]:
print("\nVehicle Columns")
print(vehicle.columns.tolist())

print("\nDisruption Columns")
print(disruption.columns.tolist())


Vehicle Columns
['RecordedTime', 'LineRef', 'Operator', 'VehicleRef', 'Latitude', 'Longitude']

Disruption Columns
['CreationTime', 'ParticipantRef', 'Progress', 'MiscellaneousReason', 'Planned', 'Summary', 'Description', 'EquipmentReason']


# Data Integration

In [17]:
from pyspark.sql.functions import broadcast
# broadcast() is used as Spark optimization where the smallar Dataframe is copied to each worker.
bus_schedule = (
    gtfs["trips"]
    .join(gtfs["routes"], "route_id", "left")
    .join(gtfs["stop_times"], "trip_id", "left")
    .join(broadcast(gtfs["stops"]), "stop_id", "left")
    .join(broadcast(gtfs["calendar"]), "service_id", "left")
)

In [18]:
print("Rows:", bus_schedule.count())
print("Columns:", len(bus_schedule.columns))

bus_schedule.printSchema()
bus_schedule.show(5, truncate=False)

Rows: 5195977
Columns: 31
root
 |-- service_id: integer (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- trip_headsign: string (nullable = true)
 |-- direction_id: integer (nullable = true)
 |-- shape_id: string (nullable = true)
 |-- wheelchair_accessible: integer (nullable = true)
 |-- agency_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_type: integer (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- timepoint: integer (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_lat: double (nullable = true)
 |-- stop_lon: double (nullable = true)
 |-- wheelchair_boarding: integer (nullable = true)
 |-

In [19]:
print("Duplicate Rows:", bus_schedule.count() - bus_schedule.dropDuplicates().count())

Duplicate Rows: 0


In [20]:
bus_schedule.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bus_schedule.columns
]).show(truncate=False)

+----------+-------+-------+--------+-------------+------------+--------+---------------------+---------+----------------+---------------+----------+------------+--------------+-------------+-----------+-------------+---------+---------+--------+--------+-------------------+------+-------+---------+--------+------+--------+------+----------+--------+
|service_id|stop_id|trip_id|route_id|trip_headsign|direction_id|shape_id|wheelchair_accessible|agency_id|route_short_name|route_long_name|route_type|arrival_time|departure_time|stop_sequence|pickup_type|drop_off_type|timepoint|stop_name|stop_lat|stop_lon|wheelchair_boarding|monday|tuesday|wednesday|thursday|friday|saturday|sunday|start_date|end_date|
+----------+-------+-------+--------+-------------+------------+--------+---------------------+---------+----------------+---------------+----------+------------+--------------+-------------+-----------+-------------+---------+---------+--------+--------+-------------------+------+-------+----

In [23]:
bus_schedule.write.mode("overwrite").parquet("Data/Processed/bus_schedule")

In [24]:
vehicle.to_csv("Data/Processed/vehicle_clean.csv", index=False)
disruption.to_csv("Data/Processed/disruption_clean.csv", index=False)

# Partitions

In [21]:
print("Partitions before:", bus_schedule.rdd.getNumPartitions())

Partitions before: 4


In [22]:
bus_schedule = bus_schedule.repartition(5)
print("Partitions after:", bus_schedule.rdd.getNumPartitions())

Partitions after: 5
